In [ ]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [ ]:
df = pd.read_csv('/content/domain_specific_chatbot_data.csv')
print(df.shape)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import train_test_split
tokenizer = T5Tokenizer.from_pretrained('t5-base')

train_data, validation_data = train_test_split(df, test_size=0.2, random_state=42)


In [ ]:
train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)

In [ ]:
train_data.head()

In [ ]:
# Clean the text by removing unwanted characters
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')


def cleaned_text(text):
    text = text.lower()
    word_tokens = word_tokenize(text)
    filtered_sentence = [word for word in word_tokens if not word in string.punctuation and word.isalnum()]
    return " ".join(filtered_sentence)


# Apply cleaning to dialogue and summary columns
train_data['query'] = train_data['query'].apply(cleaned_text)
train_data['response'] = train_data['response'].apply(cleaned_text)

validation_data['query'] = validation_data['query'].apply(cleaned_text)
validation_data['response'] = validation_data['response'].apply(cleaned_text)

In [ ]:
def preprocess_function(examples):
    inputs = tokenizer(examples["query"], padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(examples["response"], padding="max_length", truncation=True, max_length=512)
    inputs["labels"] = targets["input_ids"]
    return inputs

In [ ]:
train_dataset = train_data.apply(preprocess_function, axis=1)
val_dataset = validation_data.apply(preprocess_function,  axis=1)

In [ ]:
train_data['query'][0]

In [ ]:
train_dataset[0]

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# Model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    output_dir="./results",          # Output directory for results
    evaluation_strategy="epoch",     # Evaluate once per epoch
    save_strategy="epoch",          # Save model at the end of each epoch to match evaluation strategy
    learning_rate=3e-5,              # Learning rate
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=8,   # Batch size for evaluation
    num_train_epochs=9,              # Increase number of epochs
    weight_decay=0.01,               # Strength of weight decay
    logging_dir="./logs",            # Directory for logging
    logging_steps=10,                # Log every 10 steps
    lr_scheduler_type="linear",      # Use linear learning rate scheduler with warmup
    warmup_steps=500,                # Number of warmup steps for learning rate scheduler
    load_best_model_at_end=True,     # Load the best model at the end of training
    metric_for_best_model="eval_loss", # Monitor eval loss to determine the best model
    save_total_limit=3,              # Limit the number of checkpoints to save
    # gradient_accumulation_steps= 2   # Simulate larger batch size if GPU memory is limite
)

trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/model_chatbox2")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/model_chatbox2")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Colab Notebooks/model_chatbox2").to(device)
tokenzier = T5Tokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/model_chatbox2")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
def chatbot(query):
    query = cleaned_text(query)
    input_ids = tokenizer(query,return_tensors="pt",max_length=250,truncation=True).to(device)

    # inputs = {key: value.to(device) for key, value in input_ids.items()}

    outputs = model.generate(
        input_ids["input_ids"],
        max_length=250,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    response = chatbot(user_input)
    print("Chatbot:", response)

# You: how to login to the system?
# Chatbot: you can login to the system by visiting our website and filling out the application form.
# You: where to find setting option?
# Chatbot: to find setting option, go to the 'profile' section.
# You: How can I schedule an appointment with my doctor?
# Chatbot: you can schedule an appointment by calling our office or using our online portal.
# You: exit